In [1]:
import sqlite3
import requests
import time
import json
import pandas as pd
import urllib.parse

from dotenv import load_dotenv
import os

from pathlib import Path


In [2]:
ROOT = Path.cwd().parent


In [3]:
load_dotenv(ROOT / '.env')

TWITCH_CLIENT_ID = os.getenv('TWITCH_CLIENT_ID')
TWITCH_CLIENT_SECRET = os.getenv('TWITCH_CLIENT_SECRET')

DB_PATH = str(ROOT / 'db' / 'gaming_warehouse.db')

In [4]:
#Crear la tabla de hsitorico de reviews
def preparar_tabla_reviews():
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("""
        CREATE TABLE IF NOT EXISTS Hist_Steam_Reviews (
            resena_id TEXT PRIMARY KEY,
            juego_id INTEGER,
            resena_texto TEXT,
            recomendado INTEGER, -- (voted_up)
            votos_utiles INTEGER, -- (votes_up)
            votos_graciosos INTEGER, -- (votes_funny)
            puntuacion_ponderada REAL, -- (weighted_vote_score)
            minutos_al_resenar INTEGER, -- (playtime_at_review)
            minutos_totales INTEGER, -- (playtime_forever)
            fecha_creacion_unix INTEGER, -- (timestamp_created)
            autor_num_resenas INTEGER,
            autor_num_juegos INTEGER,
            recibido_gratis INTEGER,
            escrito_acceso_anticipado INTEGER,
            FOREIGN KEY(juego_id) REFERENCES CAT_Juego(juego_id)
        );
    """)
    conn.commit()
    conn.close()


In [5]:
preparar_tabla_reviews()

In [20]:
MAX_REVIEWS_POR_JUEGO = 1000

def obtener_juegos_pendientes():
    conn = sqlite3.connect(DB_PATH)
    query = """
        SELECT 
            j.juego_id, 
            j.id_steam, 
            j.titulo, 
            j.recommendations_count,
            COALESCE(r.reviews_en_bd, 0) as reviews_en_bd
        FROM CAT_Juego j
        LEFT JOIN (
            SELECT juego_id, COUNT(*) as reviews_en_bd 
            FROM Hist_Steam_Reviews 
            GROUP BY juego_id
        ) r ON j.juego_id = r.juego_id
        WHERE j.id_steam IS NOT NULL 
          AND COALESCE(r.reviews_en_bd, 0) < ?
        ORDER BY j.recommendations_count DESC
    """
    df = pd.read_sql_query(query, conn, params=(MAX_REVIEWS_POR_JUEGO,))
    conn.close()
    return df

In [21]:
df_juegos_pendientes=obtener_juegos_pendientes()
df_juegos_pendientes

,juego_id,id_steam,titulo,recommendations_count,reviews_en_bd
0,445,730,Counter-Strike 2,5026951.0,0
1,1,271590,Grand Theft Auto V,1873782.0,0
2,226,578080,PUBG: Battlegrounds,1758437.0,0
3,257,359550,Rainbow Six Siege,1237144.0,0
4,118,105600,Terraria,1207316.0,0
...,...,...,...,...,...
7176,10163,4230,Race: The WTCC Game,NaN,0
7177,10166,615250,M.A.X.: Mechanized Assault & Exploration,NaN,0
7178,10167,359630,Independence War 2: Edge of Chaos,NaN,0
7179,10168,44600,GTR: FIA GT Racing Game,NaN,0


In [22]:
def descargar_reviews_juego(juego_id, appid, titulo, reviews_en_bd, recommendations_count):
    reviews_objetivo = min(MAX_REVIEWS_POR_JUEGO, recommendations_count)
    reviews_restantes = reviews_objetivo - reviews_en_bd

    if reviews_restantes <= 0:
        return

    print(f"\n{titulo} (AppID: {appid})")
    print(f"  En BD: {reviews_en_bd} | Objetivo: {reviews_objetivo} | Faltan: {reviews_restantes}")

    conn = sqlite3.connect(DB_PATH)
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    cursor_bd = conn.cursor()
    cursor_api = '*'
    reviews_acumuladas = 0

    try:
        while reviews_acumuladas < reviews_restantes:
            url = (
                f"https://store.steampowered.com/appreviews/{appid}"
                f"?json=1&filter=recent&language=latam&purchase_type=all"
                f"&cursor={urllib.parse.quote(cursor_api)}&num_per_page=100"
            )

            try:
                res = requests.get(url, timeout=15)
                if res.status_code != 200:
                    print(f"  > HTTP {res.status_code}. Saltando.")
                    break

                data = res.json()
                if not data.get('success') or not data.get('reviews'):
                    print("  > Sin más reseñas en Steam.")
                    break

                batch = data['reviews']

                cursor_bd.executemany(
                    "INSERT OR IGNORE INTO Hist_Steam_Reviews VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    [
                        (
                            str(r['recommendationid']), juego_id,
                            r['review'],
                            1 if r['voted_up'] else 0,
                            r['votes_up'], r['votes_funny'],
                            float(r['weighted_vote_score']),
                            r['author']['playtime_at_review'],
                            r['author']['playtime_forever'],
                            r['timestamp_created'],
                            r['author']['num_reviews'],
                            r['author']['num_games_owned'],
                            1 if r['received_for_free'] else 0,
                            1 if r['written_during_early_access'] else 0
                        )
                        for r in batch
                    ]
                )
                conn.commit()
                reviews_acumuladas += len(batch)
                print(f"  > {reviews_acumuladas}/{reviews_restantes}")

                nuevo_cursor = data.get('cursor')
                if not nuevo_cursor or nuevo_cursor == cursor_api:
                    print("  > Fin de paginación.")
                    break
                cursor_api = nuevo_cursor

                if len(batch) < 100:
                    print("  > Último lote parcial.")
                    break

                time.sleep(0.5)

            except requests.exceptions.Timeout:
                print("time out, esperar 5 segundos")
                time.sleep(5)
            except Exception as e:
                print(f"  > Error: {e}")
                break
    finally:
        conn.close()

In [24]:
len(df_juegos_pendientes)

7181

In [26]:
total = len(df_juegos_pendientes)

In [27]:
for i, (_, juego) in enumerate(df_juegos_pendientes.iterrows(), 1):
    print(f"[{i}/{total}]", end=' ')
    descargar_reviews_juego(
        juego_id=juego['juego_id'],
        appid=juego['id_steam'],
        titulo=juego['titulo'],
        reviews_en_bd=juego['reviews_en_bd'],
        recommendations_count=juego['recommendations_count']
    )

[1/7181] 
Counter-Strike 2 (AppID: 730)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > 100/1000
  > 200/1000
  > 300/1000
  > 400/1000
  > 500/1000
  > 600/1000
  > 700/1000
  > 800/1000
  > 900/1000
  > 1000/1000
[2/7181] 
Grand Theft Auto V (AppID: 271590)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > 100/1000
  > 200/1000
  > 300/1000
  > 400/1000
  > 500/1000
  > 600/1000
  > 700/1000
  > 800/1000
  > 900/1000
  > 1000/1000
[3/7181] 
PUBG: Battlegrounds (AppID: 578080)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > 100/1000
  > 200/1000
  > 300/1000
  > 400/1000
  > 500/1000
  > 600/1000
  > 700/1000
  > 800/1000
  > 900/1000
  > 1000/1000
[4/7181] 
Rainbow Six Siege (AppID: 359550)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > 100/1000
  > 200/1000
  > 300/1000
  > 400/1000
  > 500/1000
  > 600/1000
  > 700/1000
  > 800/1000
  > 900/1000
  > 1000/1000
[5/7181] 
Terraria (AppID: 105600)
  En BD: 0 | Objetivo: 1000 | Faltan: 1000
  > 100/1000
  > 200/1000
  > 300/1000
  > 400/1000

In [30]:
conn = sqlite3.connect(DB_PATH)

resumen = pd.read_sql_query("""
    SELECT 
        COUNT(DISTINCT juego_id) as juegos_con_reviews,
        COUNT(*) as total_reviews,
        ROUND(AVG(reviews_por_juego), 1) as promedio_por_juego,
        MIN(reviews_por_juego) as min_reviews,
        MAX(reviews_por_juego) as max_reviews
    FROM (
        SELECT juego_id, COUNT(*) as reviews_por_juego
        FROM Hist_Steam_Reviews
        GROUP BY juego_id
    )
""", conn)

display(resumen)
conn.close()

,juegos_con_reviews,total_reviews,promedio_por_juego,min_reviews,max_reviews
0,5825,5825,103.0,1,1003


In [31]:
conn = sqlite3.connect(DB_PATH)

distribucion = pd.read_sql_query("""
    SELECT 
        CASE 
            WHEN reviews_por_juego >= 1000 THEN '1000+'
            WHEN reviews_por_juego >= 100  THEN '100-999'
            WHEN reviews_por_juego >= 10   THEN '10-99'
            ELSE '1-9'
        END as rango,
        COUNT(*) as juegos
    FROM (
        SELECT juego_id, COUNT(*) as reviews_por_juego
        FROM Hist_Steam_Reviews
        GROUP BY juego_id
    )
    GROUP BY rango
    ORDER BY rango DESC
""", conn)

conn.close()
display(distribucion)

,rango,juegos
0,1000+,181
1,100-999,1062
2,10-99,2676
3,1-9,1906


In [32]:
conn = sqlite3.connect(DB_PATH)

sin_reviews = pd.read_sql_query("""
    SELECT j.juego_id, j.titulo, j.recommendations_count
    FROM CAT_Juego j
    LEFT JOIN Hist_Steam_Reviews r ON j.juego_id = r.juego_id
    WHERE j.id_steam IS NOT NULL
      AND r.resena_id IS NULL
    ORDER BY j.recommendations_count DESC
""", conn)

conn.close()
print(f"Juegos sin reviews: {len(sin_reviews)}")
display(sin_reviews)

Juegos sin reviews: 1356


,juego_id,titulo,recommendations_count
0,2750,Tomb Raider: Game of the Year Edition,168219.0
1,1419,FIFA 23,123937.0
2,1169,Elden Ring: Shadow of the Erdtree,81654.0
3,18,Batman: Arkham Asylum,58656.0
4,3106,The Elder Scrolls IV: Oblivion - Game of the Y...,42255.0
...,...,...,...
1351,9709,Dead or Alive Xtreme: Venus Vacation,NaN
1352,9972,Chrome: SpecForce,NaN
1353,9978,Total War Battles: Shogun,NaN
1354,10077,Blacklight: Tango Down,NaN


In [33]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT *
        FROM CAT_Juego
        WHERE titulo like "%Batman: Arkham Asylum%"
    """, conn)
    display(df)

,juego_id,id_igdb,id_steam,titulo,categoria,fecha_lanzamiento,resumen,historia,url_portada,puntuacion_igdb,...,hltb_completacionista,steam_price_initial,steam_price_final,steam_discount_percent,metacritic_score,recommendations_count,achievements_count,steam_languages,pc_requirements_json,itad_id_texto
0,18,500,35010,Batman: Arkham Asylum,Juego principal,2009-08-25,Batman: Arkham Asylum is an action-adventure g...,"After the Joker assaults Gotham City Hall, he ...",https://images.igdb.com/igdb/image/upload/t_co...,85.669574,...,None,399.0,399.0,0,91,58656,47,"Inglés, Francés, Alemán, Italiano, Español de ...","{""minimum"": ""<ul class=\""bb_ul\""><li><br><stro...",018d937f-21ee-71f2-aa66-fe39c68e846a
1,914,27862,35140,Batman: Arkham Asylum - Game of the Year Edition,Bundle,2010-03-26,Batman: Arkham Asylum is an action-adventure g...,"After the Joker assaults Gotham City Hall, he ...",https://images.igdb.com/igdb/image/upload/t_co...,86.716050,...,None,399.0,399.0,0,91,58656,47,"Inglés, Francés, Alemán, Italiano, Español de ...","{""minimum"": ""<ul class=\""bb_ul\""><li><br><stro...",018d937f-083f-702e-9c07-4813977240a8


In [60]:
with sqlite3.connect(DB_PATH) as conn:
    df = pd.read_sql("""
        SELECT j.juego_id, j.titulo, j.id_steam, COUNT(r.resena_id) as reviews,fecha_lanzamiento,categoria
    FROM CAT_Juego j
    LEFT JOIN Hist_Steam_Reviews r ON j.juego_id = r.juego_id
    WHERE j.titulo LIKE '%Arkham%city%'
    GROUP BY j.juego_id
    """, conn)
    display(df)

,juego_id,titulo,id_steam,reviews,fecha_lanzamiento,categoria
0,17,Batman: Arkham City,57400.0,2,2011-10-18,Juego principal
1,1016,Batman: Arkham City - Game of the Year Edition,200260.0,1000,2012-05-29,Bundle
2,2291,Batman: Arkham City - Harley Quinn's Revenge,200891.0,0,2012-05-29,Expansión
3,4536,Batman: Return to Arkham - Arkham City,NaN,0,2016-10-17,Remasterización
4,5081,Batman: Arkham City - Catwoman Bundle,NaN,0,2011-10-18,Pack


los juegos faltantes de reviews son aquellos que tiene otras versiones que estana ctivas y se redirigieron ahi, steam quita versiones si ya no se ocuapran

In [58]:
def resumen_final_reviews(cantidad_top=10):
    conn = sqlite3.connect(DB_PATH)
    query = """
    SELECT 
        j.titulo,
        COUNT(r.resena_id) AS total_resenas_descargadas,
        ROUND(AVG(r.recomendado) * 100, 2) || '%' AS recomendado,
        ROUND(AVG(r.minutos_totales / 60.0), 1) AS promedio_horas_jugadas
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    GROUP BY j.juego_id
    --ORDER BY total_resenas_descargadas DESC
    LIMIT ?;
    """
    df = pd.read_sql_query(query, conn, params=(cantidad_top,))
    conn.close()
    
    display(df.sample(cantidad_top))


In [59]:
resumen_final_reviews(50)


,titulo,total_resenas_descargadas,recomendado,promedio_horas_jugadas
3,The Elder Scrolls V: Skyrim,559,97.5%,283.3
45,Cyberpunk 2077,1000,96.6%,116.0
31,Assassin's Creed IV Black Flag,432,92.36%,61.8
39,Hades,1000,99.7%,67.8
33,Max Payne,156,92.31%,13.6
43,The Elder Scrolls IV: Oblivion,272,94.85%,77.8
7,God of War,1000,99.3%,49.7
20,Hollow Knight,1000,97.7%,77.1
44,Call of Duty: Modern Warfare 2,456,97.59%,39.8
46,Mafia,178,85.39%,20.2


In [61]:
def inspeccionar_resenas(busqueda, es_id=False):
    conn = sqlite3.connect(DB_PATH)
    
    if es_id:
        filtro = "j.juego_id = ?"
        parametro = busqueda
    else:
        filtro = "j.titulo LIKE ?"
        parametro = f"%{busqueda}%"

    query = f"""
    SELECT 
        j.juego_id,
        j.titulo,
        r.recomendado,
        r.minutos_totales / 60 AS horas_jugadas,
        r.votos_utiles,
        r.resena_texto
    FROM Hist_Steam_Reviews r
    JOIN CAT_Juego j ON r.juego_id = j.juego_id
    WHERE {filtro}
    --ORDER BY r.votos_utiles DESC
    --LIMIT 10;
    """
    
    df = pd.read_sql_query(query, conn, params=(parametro,))
    conn.close()
    
    if df.empty:
        print(f"No se encontraron reseñas para: {busqueda}")
    else:
        display(df)

In [63]:
#hAbra que tratar de alguna forma las malas palabras de los jeugos ya que se ve que son creativos con las opiniones
inspeccionar_resenas("counter")


,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,445,Counter-Strike 2,1,32,0,esta divertido
1,445,Counter-Strike 2,1,136,0,"Si te odias, este juego es para vos🤙🏽"
2,445,Counter-Strike 2,1,130,0,que buen juego que es\n
3,445,Counter-Strike 2,1,70,0,Todos mis hijos
4,445,Counter-Strike 2,1,442,1,solo quiero el logro por la 1°ra reseña. mucha...
...,...,...,...,...,...,...
3720,3927,Counter-Strike Nexon,1,97,1,Ta chido
3721,3927,Counter-Strike Nexon,0,8,0,full of otakus
3722,3927,Counter-Strike Nexon,1,110,2,En todo el tiempo que lo he jugado jamas me ha...
3723,3927,Counter-Strike Nexon,1,670,0,me guta la variedad de modos de juegos \n


In [64]:
inspeccionar_resenas("pubg")

,juego_id,titulo,recomendado,horas_jugadas,votos_utiles,resena_texto
0,226,PUBG: Battlegrounds,1,738,0,ta bueno 👍
1,226,PUBG: Battlegrounds,1,751,0,exelente juego
2,226,PUBG: Battlegrounds,1,3,0,Epico siempre y cuando juegues con amigos
3,226,PUBG: Battlegrounds,0,9,0,Es agradable el juego y su gráfica pero lo que...
4,226,PUBG: Battlegrounds,1,10,0,bueb juego
...,...,...,...,...,...,...
997,226,PUBG: Battlegrounds,1,11,0,Tuvo problemas al inicio cuando lo volvieron F...
998,226,PUBG: Battlegrounds,1,679,0,"DE LO MEJOR, MUY REALISTA"
999,226,PUBG: Battlegrounds,1,1091,0,MEDIO PE
1000,226,PUBG: Battlegrounds,1,463,0,Probablemente de los unicos battle royale que ...
